# SPECTER Embeddings — Corpus, Queries, Queries_1

Generates SPECTER embeddings for all three datasets with:
- **GPU/TPU acceleration** (auto-detected)
- **Resumable** — picks up from last saved batch
- Saves embeddings as `.npy` files per dataset

In [3]:
# ── Install dependencies ──────────────────────────────────────────────────────
# Pin huggingface_hub<0.23 to avoid HfFolder removal incompatibility with adapters
!pip install -q "huggingface_hub==0.22.2"
!pip install -q transformers accelerate "adapters>=1.0.0"


In [4]:
import os, json, math, time
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from transformers import AutoTokenizer
from adapters import AutoAdapterModel

# ── Paths — adjust INPUT_DIR if your Kaggle dataset is mounted elsewhere ──────
INPUT_DIR  = Path("/Users/olutolaoloruntobipaul/Downloads/ir_challenge/data")   # <-- change to your dataset path
OUTPUT_DIR = Path("../submissions")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Device ────────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device("cpu")
    print("No GPU found — running on CPU (will be slow)")

# ── Config ────────────────────────────────────────────────────────────────────
BASE_MODEL   = "allenai/specter2_base"
ADAPTER_NAME = "allenai/specter2_proximity"
BATCH_SIZE   = 32                  # lower to 8–16 if you hit OOM on small GPU
MAX_LENGTH   = 512
SAVE_EVERY   = 200                 # save a checkpoint every N batches

print(f"Output dir  : {OUTPUT_DIR}")
print(f"Base model  : {BASE_MODEL}")
print(f"Adapter     : {ADAPTER_NAME}")
print(f"Batch size  : {BATCH_SIZE}")

No GPU found — running on CPU (will be slow)
Output dir  : ../submissions
Base model  : allenai/specter2_base
Adapter     : allenai/specter2_proximity
Batch size  : 32


In [5]:
# ── Load tokenizer + model (SPECTER2 with proximity adapter) ────────────────
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoAdapterModel.from_pretrained(BASE_MODEL)
model.load_adapter(ADAPTER_NAME, source="hf", load_as="specter2_proximity", set_active=True)
model = model.to(DEVICE)
model.eval()
print("Model + adapter loaded.")

Fetching 4 files: 100%|██████████| 4/4 [00:02<00:00,  1.67it/s]
There are adapters available but none are activated for the forward pass.


Model + adapter loaded.


In [6]:
# ── Helper: embed a list of texts, returns (N, 768) float32 numpy array ───────
@torch.no_grad()
def embed_texts(texts: list[str]) -> np.ndarray:
    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    ).to(DEVICE)
    out = model(**encoded)
    # CLS token representation — same as SPECTER paper
    vecs = out.last_hidden_state[:, 0, :]
    return vecs.cpu().float().numpy()

In [7]:
# ── Core embedding loop with checkpointing ────────────────────────────────────
def embed_dataset(
    df: pd.DataFrame,
    name: str,
    title_col: str = "title",
    abstract_col: str = "abstract",
) -> np.ndarray:
    """
    Embed all rows in `df`.  Saves partial results every SAVE_EVERY batches so
    the run can be resumed if it crashes or is interrupted.

    Resume logic:
      - Checkpoint file  : OUTPUT_DIR/<name>_checkpoint.npz
      - Final output     : OUTPUT_DIR/<name>_embeddings.npy  (N, 768)
      - ID file          : OUTPUT_DIR/<name>_ids.json
    """
    final_emb_path = OUTPUT_DIR / f"{name}_embeddings.npy"
    final_ids_path = OUTPUT_DIR / f"{name}_ids.json"
    ckpt_path      = OUTPUT_DIR / f"{name}_checkpoint.npz"

    if final_emb_path.exists() and final_ids_path.exists():
        print(f"[{name}] Already done — skipping.")
        return np.load(final_emb_path)

    n_docs = len(df)
    ids    = df["doc_id"].tolist()

    # SPECTER2 input format: title [SEP] abstract
    sep   = tokenizer.sep_token or "[SEP]"
    texts = [
        f"{str(row[title_col])} {sep} {str(row[abstract_col])}"
        for _, row in df.iterrows()
    ]

    if ckpt_path.exists():
        ckpt      = np.load(ckpt_path, allow_pickle=True)
        done_embs = list(ckpt["embeddings"])
        start_idx = int(ckpt["start_idx"])
        print(f"[{name}] Resuming from {start_idx}/{n_docs}")
    else:
        done_embs = []
        start_idx = 0
        print(f"[{name}] Starting — {n_docs} documents")

    batches_done = 0
    pbar = tqdm(range(start_idx, n_docs, BATCH_SIZE), desc=name, unit="batch")

    for batch_start in pbar:
        batch_end   = min(batch_start + BATCH_SIZE, n_docs)
        batch_texts = texts[batch_start:batch_end]
        batch_embs  = embed_texts(batch_texts)
        done_embs.append(batch_embs)
        batches_done += 1

        pbar.set_postfix(docs=batch_end, total=n_docs)

        if batches_done % SAVE_EVERY == 0 and batch_end < n_docs:
            np.savez(
                ckpt_path,
                embeddings=np.concatenate(done_embs, axis=0),
                start_idx=np.array(batch_end)
            )
            tqdm.write(f"[{name}] Checkpoint saved at {batch_end}")

    all_embs = np.concatenate(done_embs, axis=0)
    np.save(final_emb_path, all_embs)
    with open(final_ids_path, "w") as f:
        json.dump(ids, f)

    if ckpt_path.exists():
        ckpt_path.unlink()

    print(f"[{name}] DONE — shape={all_embs.shape}")
    return all_embs

In [8]:
# ── Load data ─────────────────────────────────────────────────────────────────
corpus    = pd.read_parquet(INPUT_DIR / "corpus.parquet")
queries   = pd.read_parquet(INPUT_DIR / "queries.parquet")
queries_1 = pd.read_parquet(INPUT_DIR / "queries_1.parquet")

print(f"corpus    : {len(corpus):,} rows")
print(f"queries   : {len(queries):,} rows")
print(f"queries_1 : {len(queries_1):,} rows")

corpus    : 20,000 rows
queries   : 100 rows
queries_1 : 100 rows


In [9]:
# ── Embed corpus (largest — ~20k docs) ───────────────────────────────────────
corpus_embs = embed_dataset(corpus, name="corpus")

[corpus] Starting from scratch — 20000 documents
[corpus] 320/20000  (1.6%)  5 docs/s  ETA 3707s
[corpus] 640/20000  (3.2%)  5 docs/s  ETA 3790s
[corpus] 960/20000  (4.8%)  5 docs/s  ETA 3731s
[corpus] 1280/20000  (6.4%)  5 docs/s  ETA 3643s
[corpus] 1600/20000  (8.0%)  5 docs/s  ETA 3563s
[corpus] 1920/20000  (9.6%)  5 docs/s  ETA 3542s


KeyboardInterrupt: 

In [ ]:
# ── Embed queries ─────────────────────────────────────────────────────────────
queries_embs = embed_dataset(queries, name="queries")

In [ ]:
# ── Embed queries_1 ───────────────────────────────────────────────────────────
queries_1_embs = embed_dataset(queries_1, name="queries_1")

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
print("\n=== All embeddings generated ===")
for name in ["corpus", "queries", "queries_1"]:
    emb_path = OUTPUT_DIR / f"{name}_embeddings.npy"
    ids_path = OUTPUT_DIR / f"{name}_ids.json"
    if emb_path.exists():
        arr = np.load(emb_path)
        with open(ids_path) as f:
            ids = json.load(f)
        print(f"  {name:12s}: shape={arr.shape}  ids={len(ids)}")

print(f"\nOutput directory contents:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {p.name:50s} {p.stat().st_size/1e6:.1f} MB")

In [ ]:
# ── Encode queries with FULL_TEXT field (for better citation retrieval) ──────
# Currently the default encoding uses title+abstract (ta).
# Using full_text captures methods/results/references — stronger citation signal.

def get_ft_text(df, ids, max_chars=4096):
    """Get full_text for each doc_id, fallback to ta/title+abstract."""
    texts = []
    for did in ids:
        row = df.loc[did]
        ft = row.get("full_text", "")
        if pd.notna(ft) and str(ft).strip():
            texts.append(str(ft)[:max_chars])
        else:
            ta = row.get("ta", "")
            if pd.notna(ta) and str(ta).strip():
                texts.append(str(ta))
            else:
                t = str(row.get("title", "") or "")
                a = str(row.get("abstract", "") or "")
                texts.append(f"{t} {a}".strip())
    return texts

@torch.no_grad()
def encode_texts_ft(texts, batch_size=16):
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(
            batch, padding=True, truncation=True,
            return_tensors="pt", max_length=512
        ).to(DEVICE)
        out = model(**enc)
        emb = out.last_hidden_state[:, 0, :].cpu().float().numpy()
        all_embs.append(emb)
    return np.vstack(all_embs)

# Encode val queries (queries_1) with full_text
queries_1_df = queries_1.set_index("doc_id")
queries_df   = queries.set_index("doc_id")

val_qids   = json.load(open(str(INPUT_DIR / "embeddings/sentence-transformers_all-MiniLM-L6-v2/query_ids.json")))
held_qids  = queries["doc_id"].tolist()

print("Encoding val queries with full_text...")
val_ft_texts  = get_ft_text(queries_1_df, val_qids)
val_ft_emb    = encode_texts_ft(val_ft_texts)
print(f"val_ft_emb shape: {val_ft_emb.shape}")
np.save(OUTPUT_DIR / "specter2_prox_val_queries_ft.npy", val_ft_emb)

print("Encoding held-out queries with full_text...")
held_ft_texts = get_ft_text(queries_df, held_qids)
held_ft_emb   = encode_texts_ft(held_ft_texts)
print(f"held_ft_emb shape: {held_ft_emb.shape}")
np.save(OUTPUT_DIR / "specter2_prox_heldout_queries_ft.npy", held_ft_emb)

print("Done! Download specter2_prox_val_queries_ft.npy and specter2_prox_heldout_queries_ft.npy")


## Quick sanity check — cosine similarity between a query and a few corpus docs

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Reload
c_embs  = np.load(OUTPUT_DIR / "corpus_embeddings.npy")
q_embs  = np.load(OUTPUT_DIR / "queries_embeddings.npy")

# Top-5 corpus docs for first query
sims    = cosine_similarity(q_embs[[0]], c_embs)[0]
top5    = np.argsort(sims)[::-1][:5]

with open(OUTPUT_DIR / "corpus_ids.json") as f:
    corpus_ids = json.load(f)
with open(OUTPUT_DIR / "queries_ids.json") as f:
    query_ids = json.load(f)

print(f"Query id  : {query_ids[0]}")
print(f"Top-5 corpus matches:")
for rank, idx in enumerate(top5, 1):
    print(f"  {rank}. {corpus_ids[idx]}  sim={sims[idx]:.4f}")